# DINO AQUA20 Training Launcher

Launches `main_dino_aqua.py` via `subprocess.Popen`, streams output live to this notebook,
and saves a full log to `{output_dir}/train.log`.

**Note**: Interrupting the kernel does **not** kill the training process — it runs independently.
To kill it after interrupting: `import os, signal; os.kill(<PID>, signal.SIGTERM)`

In [ ]:
import subprocess
import os
import sys
import signal
import datetime
from pathlib import Path

Run name : dino-aqua20-20260505_120658
Output   : /home/alex/internship/dino/outputs/dino-aqua20-20260505_120658
W&B name : dino-aqua20-20260505_120658


In [ ]:
# ── Run configuration ───────────────────────────────────────────────────────
REPO_DIR = Path("/home/alex/internship/dino").resolve()
DATA_PATH = (
    "/home/alex/internship/GradientDistillation/logged_files/distillation/"
    "aqua20/dinov2_vitb/dinov2_vitb_distill_196_ipc1_augs3/data.pth"
)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name  = f"dino-aqua20-{timestamp}"
output_dir = REPO_DIR / "outputs" / run_name
output_dir.mkdir(parents=True, exist_ok=True)

HP = dict(
    arch                        = "vit_small",
    patch_size                  = 16,
    out_dim                     = 65536,
    norm_last_layer             = "true",
    momentum_teacher            = 0.996,
    warmup_teacher_temp         = 0.04,
    teacher_temp                = 0.07,
    warmup_teacher_temp_epochs  = 30,
    use_fp16                    = "true",
    weight_decay                = 0.04,
    weight_decay_end            = 0.4,
    clip_grad                   = 3.0,
    batch_size_per_gpu          = 64,
    epochs                      = 100,
    warmup_epochs               = 10,
    lr                          = 0.0005,
    min_lr                      = 1e-6,
    optimizer                   = "adamw",
    drop_path_rate              = 0.1,
    freeze_last_layer           = 1,
    global_crops_scale          = "0.4 1.0",
    local_crops_number          = 8,
    local_crops_scale           = "0.05 0.4",
    num_workers                 = 4,
    saveckp_freq                = 20,
    seed                        = 0,
    # kNN eval
    knn_eval_freq               = 10,
    knn_nb_knn                  = "10 20 100 200",
    knn_temperature             = 0.07,
    num_classes                 = 20,
    # W&B
    use_wandb                   = "true",
    wandb_project               = "dino-aqua20",
    wandb_run_name              = run_name,
)

print(f"Run name : {run_name}")
print(f"Output   : {output_dir}")
print(f"W&B name : {HP['wandb_run_name']}")

Command built (see launched command in train.log header)


In [ ]:
# ── Build command ────────────────────────────────────────────────────────────
# Multi-value args (lists) are passed as space-separated strings in HP;
# we split them when building the cmd list.
MULTI_VALUE_ARGS = {"global_crops_scale", "local_crops_scale", "knn_nb_knn"}

cmd = [
    "uv", "run",
    "-m", "torch.distributed.launch",
    "--nproc_per_node=1",
    "main_dino_aqua.py",
    "--distilled_data_path", DATA_PATH,
    "--output_dir", str(output_dir),
]
# Add dist_url with unique file to avoid collisions with other runs
cmd += ["--dist_url", f"file:///tmp/dino_dist_{timestamp}"]

for key, val in HP.items():
    if key in MULTI_VALUE_ARGS:
        cmd += [f"--{key}"] + str(val).split()
    else:
        cmd += [f"--{key}", str(val)]

print("Command:")
print(" \\".join(["  " + c for c in cmd]))

Started PID=15786  log=/home/alex/internship/dino/outputs/dino-aqua20-20260505_120658/train.log
wandb: Syncing run dino-aqua20-20260505_120658
wandb: View run at https://wandb.ai/alex26delaveau-lyon-2-/dino-aqua20/runs/0zvrv54f
Data loaded: there are 200 images.
Student and Teacher are built: they are both vit_small network.
Starting DINO training !
Epoch: [0/100] Total time: 0:00:59  loss: 10.796501
Epoch: [1/100] Total time: 0:00:21  loss: 10.854280
Epoch: [2/100] Total time: 0:00:22  loss: 10.958606
(training confirmed running, streaming loop stopped for notebook save)


In [ ]:
# ── Launch training and stream output ────────────────────────────────────────
# stdout + stderr are merged and streamed line-by-line:
#   - printed to this cell (visible in notebook)
#   - written to {output_dir}/train.log
#
# Interrupting the kernel stops the streaming loop but NOT the subprocess.
# The process keeps running and logs keep accumulating in train.log.

log_path = output_dir / "train.log"

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = "/usr/lib/wsl/lib:" + env.get("LD_LIBRARY_PATH", "")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=str(REPO_DIR),
    env=env,
)

print(f"Started PID={process.pid}  log={log_path}")
print("=" * 70)

with open(log_path, "w") as log_file:
    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

process.wait()
print(f"\nProcess exited with code {process.returncode}")

PID         : 15786
Output dir  : /home/alex/internship/dino/outputs/dino-aqua20-20260505_120658
Log file    : /home/alex/internship/dino/outputs/dino-aqua20-20260505_120658/train.log
W&B project : dino-aqua20
W&B run     : dino-aqua20-20260505_120658

To tail the log from a terminal:
  tail -f /home/alex/internship/dino/outputs/dino-aqua20-20260505_120658/train.log

To kill the run if kernel was interrupted:
  import os, signal; os.kill(15786, signal.SIGTERM)


In [ ]:
# ── Run info & kill instructions ─────────────────────────────────────────────
print(f"PID         : {process.pid}")
print(f"Output dir  : {output_dir}")
print(f"Log file    : {log_path}")
print(f"W&B project : {HP['wandb_project']}")
print(f"W&B run     : {HP['wandb_run_name']}")
print()
print("To tail the log from a terminal:")
print(f"  tail -f {log_path}")
print()
print("To kill the run if kernel was interrupted:")
print(f"  import os, signal; os.kill({process.pid}, signal.SIGTERM)")